# Surgical RAG-VQA — Spike Analysis Notebook

Interactive analysis of feasibility spike results.
Run this after `python scripts/run_all.py`.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, Image, Markdown

sns.set_theme(style='whitegrid', font_scale=1.1)
PROJECT = Path('.').resolve()
RESULTS = PROJECT / 'results'
FRAMES = PROJECT / 'data' / 'frames'

print(f'Project root: {PROJECT}')

## 1. Load Results

In [ ]:
# Try full results first, fall back to mock
results_path = RESULTS / 'spike_results_v1.json'
if not results_path.exists():
    results_path = RESULTS / 'mock_retrieval_results.json'

with open(results_path) as f:
    results = json.load(f)

df = pd.DataFrame(results)
print(f'Loaded {len(df)} results from {results_path.name}')
df[['qid', 'question_type', 'difficulty', 'is_defer', 'confidence', 'should_defer']].head(10)

## 2. Defer Mechanism Analysis

In [ ]:
if 'is_defer' in df.columns:
    # Confusion matrix for defer
    df['defer_correct'] = df.apply(
        lambda r: 'TP' if r['should_defer'] and r.get('is_defer') else
                  'TN' if not r['should_defer'] and not r.get('is_defer') else
                  'FP' if not r['should_defer'] and r.get('is_defer') else 'FN',
        axis=1
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Defer distribution
    defer_counts = df['defer_correct'].value_counts()
    colors = {'TP': '#2ecc71', 'TN': '#3498db', 'FP': '#f39c12', 'FN': '#e74c3c'}
    defer_counts.plot.bar(
        ax=axes[0],
        color=[colors.get(x, '#95a5a6') for x in defer_counts.index]
    )
    axes[0].set_title('Defer Mechanism — Confusion Matrix')
    axes[0].set_xlabel('')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=0)
    
    # By question type
    type_defer = df.groupby('question_type')['is_defer'].agg(['sum', 'count'])
    type_defer.columns = ['deferred', 'total']
    type_defer['answered'] = type_defer['total'] - type_defer['deferred']
    type_defer[['answered', 'deferred']].plot.bar(
        stacked=True, ax=axes[1],
        color=['#3498db', '#e74c3c']
    )
    axes[1].set_title('Answer vs Defer — by Question Type')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(RESULTS / 'defer_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No defer data (mock mode) — run full pipeline for this analysis')

## 3. Confidence Distribution

In [ ]:
if 'confidence' in df.columns and df['confidence'].notna().any():
    fig, ax = plt.subplots(figsize=(8, 5))
    
    conf_order = ['high', 'medium', 'low', 'unknown']
    conf_colors = {'high': '#2ecc71', 'medium': '#f39c12', 'low': '#e74c3c', 'unknown': '#95a5a6'}
    
    conf_counts = df['confidence'].value_counts().reindex(conf_order, fill_value=0)
    conf_counts.plot.bar(
        ax=ax,
        color=[conf_colors.get(x, '#95a5a6') for x in conf_counts.index]
    )
    ax.set_title('VLM Confidence Distribution')
    ax.set_xlabel('Confidence Level')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.savefig(RESULTS / 'confidence_dist.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No confidence data available')

## 4. Retrieved Chunks Analysis

In [ ]:
# Check retrieval quality
if 'retrieved_chunks' in df.columns:
    print('Retrieved chunks per question:\n')
    for _, row in df.iterrows():
        chunks = row.get('retrieved_chunks', [])
        scores = row.get('retrieved_scores', [])
        print(f"  {row['qid']}: {len(chunks)} chunks")
        for c, s in zip(chunks, scores):
            print(f"    └─ {c}  (score: {s})")
else:
    print('No retrieval data available')

## 5. Detailed Per-Question Review

In [ ]:
for _, row in df.iterrows():
    qid = row['qid']
    print(f"\n{'='*60}")
    print(f"{qid} [{row['question_type']}] — {row.get('difficulty', '?')}")
    print(f"Q: {row.get('question', 'N/A')}")
    print(f"Gold: {row.get('gold_answer', 'N/A')}")
    
    if 'parsed_answer' in row and pd.notna(row.get('parsed_answer')):
        print(f"Model: {row['parsed_answer']}")
    if 'is_defer' in row:
        print(f"Deferred: {row['is_defer']}  |  Should defer: {row['should_defer']}")
    if 'confidence' in row:
        print(f"Confidence: {row.get('confidence', '?')}")
    
    # Show frame if available
    frame_path = FRAMES / row.get('frame', '')
    if frame_path.exists():
        display(Image(filename=str(frame_path), width=400))

## 6. Summary Table for Report

In [ ]:
summary_cols = ['qid', 'question_type', 'difficulty']
if 'is_defer' in df.columns:
    summary_cols += ['is_defer', 'should_defer', 'confidence']

summary = df[summary_cols].copy()
display(summary.style.set_caption('Spike Results Summary'))